# tpattern — the worked example, step by step

This notebook is the companion to the *Behavior Research Methods* tutorial: it runs the **worked example (Section 4)** end to end, in the browser, with nothing to install locally. Run the cells in order.

T-pattern analysis applies to **any** stream of timed, categorised events — behavioural coding, animal behaviour, clinical or physiological logs, human–computer interaction, conversation. Here the worked instance is the 2022 World Cup goal sequences; to analyse your own data, replace the input table in Step 1 and re-run.

**What you will do:** load events → inspect and diagnose → let the tool recommend the settings → calibrate against a chance model → read the result and its auto-generated Methods paragraph.

## Step 0 — Install and first run

Install the package and get the shipped data and examples, then run a first analysis on the small **synthetic** dataset so you reach a result immediately.

In [ ]:
#@title Install tpattern + fetch data/examples { display-mode: "form" }
!git clone -q https://github.com/ajcallaway/TPattern.git
%pip install -q tpattern   # library from PyPI (repo cloned above only for the example data)
print('\nStep 0 — first run on the shipped synthetic ground-truth data:')
!python TPattern/examples/quickstart.py

## Step 1 — Load

Read the events from a single flat table (one row per event). Naming the observation-window columns means each sequence's window length *T* is taken from the data, not inferred from the first and last event.

In [ ]:
from tpattern import read_table

observations = read_table("TPattern/data/worldcup_goals.csv",
                          obs_start="obs_start", obs_end="obs_end", time_unit="ms")
print(f"{len(observations)} sequences, "
      f"{sum(len(o.events) for o in observations)} events")

## Step 2 — Inspect

The tool tests whether the analytic assumptions hold (conditional uniformity, dispersion). Where they fail — as they do here — it flags that the analytic *p* must not be thresholded directly and that surrogate calibration is required.

In [ ]:
from tpattern import recommend

rec = recommend(observations)
print(rec)   # dataset summary + the three recommended settings, with the reasons

## Step 3 — Recommendation and Methods text

From the same measurements the advisor recommends the surrogate null, the minimum lag, and the error-rate control — and writes the Methods sentence that justifies each, straight from the data.

In [ ]:
print(rec.methods_text())

## Step 4 — Run and read

Detect and calibrate under the recommended settings (profile-preserving null, minimum lag of one frame, *B* = 2,000 surrogates — a few minutes). Every detected pattern is tested against the chance model with false-discovery correction; the survivors are the reportable result.

In [ ]:
from tpattern import Config, calibrate, report

result = calibrate(observations, Config(min_lag=1),
                   null="profile", B=2000, q_target=0.05)

survivors = sorted(result.kept("fdr"), key=lambda c: c.fdr_q)
print(f"{len([c for c in result.real if c.level >= 1])} composite patterns detected; "
      f"{len(survivors)} survive false-discovery control:\n")
for c in survivors:
    print(f"  {c.pattern.signature():55s}  N={c.N:2d}  p={c.p_emp:.4f}  q={c.fdr_q:.3f}")

report(result, "goals_out", title="Goals")

In [ ]:
#@title Show the pattern dendrogram and the generated report { display-mode: "form" }
from IPython.display import Image, display
import os
for f in ("patterns_overview.png",):
    p = os.path.join("goals_out", f)
    if os.path.exists(p):
        display(Image(p))
print(open("goals_out/SUMMARY.txt").read())

## Step 5 — Every setting you can change, and who decides it

Nothing in this analysis is hidden from you. The settings fall into three groups, and the
difference between them matters more than any individual value:

* **A. The data can decide these.** `recommend()` sets them from measurements of your own data
  and tells you why (Step 3). You can override any of them.
* **B. The data cannot decide these.** They are your policy. The tool gives a default and tells
  you what changing it costs, but it will not choose for you.
* **C. Structural inputs.** Fixed before detection begins. These never present themselves as
  decisions, which is exactly why they are worth stating: the observation window in particular
  sets the length *T* used as the denominator of the baseline, so getting it wrong shifts every
  probability in the analysis.

Change any field below and re-run this cell and the next.

In [ ]:
#@title Step 5 — the full settings panel { display-mode: "form" }

#@markdown ### A. The data can decide these (see Step 3)
surrogate_null = "profile"  #@param ["profile", "rotation", "shuffle"]
minimum_lag_frames = 1  #@param {type:"integer"}
frequent_event_exclusion = 1.5  #@param {type:"number"}
minimum_occurrence = 3  #@param {type:"integer"}
error_control = "fdr"  #@param ["fdr", "fwer"]
surrogates_B = 2000  #@param {type:"integer"}

#@markdown ### B. Your policy — the data cannot decide these
#@markdown `bout_support_percent = 0` means off. Step 6 works through what happens if you raise it.
detection_alpha = 0.005  #@param {type:"number"}
false_discovery_q = 0.05  #@param {type:"number"}
family_wise_alpha = 0.005  #@param {type:"number"}
bout_support_percent = 0.0  #@param {type:"number"}
lumping_factor = 0.0  #@param {type:"number"}
completeness_competition = True  #@param {type:"boolean"}
collapse_equivalent_patterns = True  #@param {type:"boolean"}
include_univariate_patterns = True  #@param {type:"boolean"}
force_exclude_events = ""  #@param {type:"string"}

#@markdown ### C. Structural inputs — these change every result
time_unit = "ms"  #@param ["s", "ms"]
use_observation_window_columns = True  #@param {type:"boolean"}
maximum_depth = 8  #@param {type:"integer"}
candidate_edge_cap = 0  #@param {type:"integer"}

from tpattern import Config, read_table

observations = read_table(
    "TPattern/data/worldcup_goals.csv",
    obs_start="obs_start" if use_observation_window_columns else None,
    obs_end="obs_end" if use_observation_window_columns else None,
    time_unit=time_unit)

cfg = Config(
    alpha=detection_alpha,
    min_occurrence=minimum_occurrence,
    min_lag=minimum_lag_frames,
    freq_exclude=frequent_event_exclusion,
    exclude_events=[e.strip() for e in force_exclude_events.split(",") if e.strip()] or None,
    include_univariate=include_univariate_patterns,
    completeness=completeness_competition,
    collapse_equivalent=collapse_equivalent_patterns,
    lumping_factor=lumping_factor or None,
    min_samples_frac=(bout_support_percent / 100) or None,
    max_edges=candidate_edge_cap or None,
    max_level=maximum_depth,
)

print(f"{len(observations)} observations (bouts) loaded.")
print("\nDetection settings in force:")
for k, v in vars(cfg).items():
    print(f"  {k:24s} {v}")


## Step 6 — Working through a setting the data cannot choose for you

Step 5 left `bout_support_percent` at 0, i.e. off. Bout support says: *a pattern must appear in
at least this fraction of bouts to be kept*. A value like 10% or 25% sounds like a sensible,
conservative safeguard. This step works through whether it is.

Three questions, in order. Answer them for any Class B setting, not just this one.

**1. What is the question?**
"Should I require a pattern to appear in at least 10% of bouts before I report it?"

**2. What happens if I do?**
Run the next cell. It calibrates once with no support threshold, then reports, for each pattern
that survived the chance test, the fraction of bouts it actually spans. That tells you directly
which of your real results a threshold would delete.

**3. What is the defensible answer?**
Read it off the output, then see the note below the cell.

In [ ]:
#@title Step 6 — what would a bout-support threshold cost you? { display-mode: "form" }
from tpattern import calibrate, patterns_table, support_profile

n_bouts = len(observations)
result = calibrate(observations, cfg, null=surrogate_null, B=surrogates_B,
                   alpha=family_wise_alpha, q_target=false_discovery_q, seed=20260714)
survivors = sorted(result.kept(error_control), key=lambda c: c.fdr_q)

print(f"{n_bouts} bouts. {len(survivors)} pattern(s) survive with NO support threshold.\n")
print("Prevalence is REPORTED, not filtered on. Every results row carries `bouts` and")
print("`bout_support`, so you can apply any threshold yourself -- and see what it costs.\n")

supports = []
for c in survivors:
    b = len({i.obs for i in c.pattern.instances})
    s = b / n_bouts
    supports.append(s)
    print(f"  {c.pattern.signature():<52s} N={c.N:>3d}  {b:>3d}/{n_bouts} bouts = {s:>5.1%}  q={c.fdr_q:.3f}")

if supports:
    print(f"\nLowest support among survivors: {min(supports):.1%}")
    print("Below that a threshold changes nothing; above it, it starts deleting")
    print("patterns that passed the chance test.\n")
    for pct in (2, 5, 10, 25):
        lost = sum(1 for s in supports if s < pct / 100)
        flag = "" if lost == 0 else f"   <-- deletes {lost}"
        print(f"  a {pct:>2d}% threshold keeps {len(supports)-lost} of {len(supports)}{flag}")
    support_profile(result, "support_profile.png",
                    title="Prevalence of the calibrated patterns",
                    mark_thresholds=(0.10,))
    print("\nWrote support_profile.png")
else:
    floor = 1 / (surrogates_B + 1)
    print("No pattern cleared correction, so there is nothing for a support threshold to delete.")
    print(f"With B = {surrogates_B} the empirical p cannot fall below 1/(B+1) = {floor:.4f}.")
    print("If that floor is above what the correction requires for this family, a genuine")
    print("pattern cannot resolve however real it is. Raise surrogates_B in Step 5 and re-run;")
    print("on these goal data B = 2000 is enough and B = 200 is not.")

#@markdown ---
#@markdown This filters the survivors by the support they actually have, which is the direct
#@markdown way to see what a threshold would remove. Re-running detection with the threshold
#@markdown set can differ slightly, because pruning during the search also shrinks the family
#@markdown of patterns being corrected for.


### Reading the result

The pattern this analysis exists to report is the cross delivered one to five seconds before the
shot. It is genuine: it survives calibration against the chance model. But it appears in only a
small minority of goal sequences, because most goals do not come from a cross. **A threshold set
above that prevalence deletes it, and you are never told.**

The larger non-goals group makes this starker. Eighteen patterns survive calibration there, and
every one of them sits below 10% support, ranging from 8.2% down to 0.2%. A 10% threshold removes
all eighteen.

**Support and significance are different axes.** Support asks *how widespread?*; calibration asks
*is this more than chance?* A pattern can be strongly non-random and confined to a minority of
bouts — the normal case for anything specific to a subgroup. In the non-goals set the two axes are
close to unrelated: several patterns at 0.5% support carry stronger evidence (q = .010) than the
two most widespread ones (q = .031).

**Would an elbow help?** `support_profile.png` has a visible knee: two patterns near 8.2%, then a
drop to 1.9% and a long tail. It is tempting to cut there, the way a scree plot or a filter
residual curve is read in engineering. Those work because the bend separates signal from noise.
Here it cannot: calibration has *already* separated signal from noise, so the bend separates only
common from rare. Cutting at roughly 2% would discard sixteen of the eighteen patterns, including
the best-evidenced ones. Nor does the bend mark a boundary between kinds of play: the two most widespread
patterns are both cross-delivery, and although every looseball chain sits in the low-prevalence
tail, that tail also contains cross and open-play patterns throughout. The bend marks where the
two commonest patterns stop — a fact about prevalence, not about football.

**So what should you do?** Report prevalence instead of filtering on it. The results table carries
`bouts` and `bout_support` for every pattern, so a reader sees the spread directly and applies
whatever bar they think right. A threshold applied in advance takes that choice away and destroys
the information it acted on.

Two reasons remain to set one at all:

1. **Computation.** Support prunes the search and may be what makes a dense analysis finish. Then
   say which patterns were never enumerated.
2. **A genuine prevalence hypothesis.** If the question really is "which patterns are typical of at
   least a quarter of bouts?", 25% *is* the question, fixed in advance and part of the hypothesis.

Outside those two, leave it off.

**What makes it circular.** Choosing the threshold by seeing which value gives the nicest
downstream result is the failure described in Section 2. The threshold is a researcher degree of
freedom; selected on the outcome, the outcome is no longer trustworthy — and an elbow rule does not
repair that, it only makes the choice look principled.

## Now run it on your own data

Replace the file in **Step 1** with your own flat event table — one row per event, with an `observation` id, an `event` label, and a `start` time (add `obs_start`/`obs_end` if you have explicit windows) — and re-run every cell. The diagnostics, the recommended settings, the calibration and the Methods text all regenerate from whatever data you supply.

See [`SCHEMA.md`](https://github.com/ajcallaway/TPattern/blob/main/SCHEMA.md) for the (simple) input format.